In [ ]:
from transformers.models.bert import BertTokenizer,BertModel
import torch

In [ ]:
model_type = 'bert-base-uncased'

In [ ]:
bert = BertModel.from_pretrained(model_type)
tokenizer = BertTokenizer.from_pretrained(model_type)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
text = 'this is a text sentence.'

In [ ]:
inputs = tokenizer(text,return_tensors='pt')
inputs

{'input_ids': tensor([[ 101, 2023, 2003, 1037, 3793, 6251, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}

In [ ]:
tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

['[CLS]', 'this', 'is', 'a', 'text', 'sentence', '.', '[SEP]']

# forward and pooler output

In [ ]:
bert.eval()
with torch.no_grad():
  output = bert(**inputs)

In [ ]:
output.keys()

odict_keys(['last_hidden_state', 'pooler_output'])

In [ ]:
output['last_hidden_state'].shape

torch.Size([1, 8, 768])

In [ ]:
output['pooler_output']

tensor([[-0.9320, -0.4660, -0.7054,  0.8013,  0.5395, -0.2326,  0.8985,  0.3100,
         -0.5941, -1.0000, -0.1055,  0.8209,  0.9858,  0.2604,  0.9526, -0.6847,
         -0.2648, -0.6366,  0.3471, -0.7114,  0.6518,  0.9998,  0.4752,  0.3604,
          0.5276,  0.9401, -0.6821,  0.9463,  0.9627,  0.7596, -0.7972,  0.2211,
         -0.9909, -0.2564, -0.7488, -0.9928,  0.4347, -0.8085, -0.0717, -0.0137,
         -0.9286,  0.3592,  1.0000, -0.4739,  0.2884, -0.4205, -1.0000,  0.3234,
         -0.9190,  0.7232,  0.6715,  0.5167,  0.2301,  0.5236,  0.5461, -0.0622,
         -0.0630,  0.1883, -0.2802, -0.6607, -0.6670,  0.3900, -0.6133, -0.9413,
          0.5821,  0.5472, -0.1604, -0.3769, -0.1576, -0.0250,  0.9041,  0.2955,
         -0.0418, -0.8321,  0.3900,  0.3109, -0.6546,  1.0000, -0.5625, -0.9819,
          0.6171,  0.5162,  0.6011,  0.0482,  0.2289, -1.0000,  0.5958, -0.2103,
         -0.9917,  0.1602,  0.5617, -0.2821,  0.4634,  0.6287, -0.4933, -0.3350,
         -0.4028, -0.5878, -

In [ ]:
output['pooler_output'].shape

torch.Size([1, 768])

# from scratch

提取 [CLS] 词元的隐藏状态
代码片段：output['last_hidden_state'][0][0, :]

output['last_hidden_state']：这是模型最后一层输出的所有词元的特征矩阵。通常形状是 [batch_size, sequence_length, hidden_size]（比如 [1, 10, 768]）。

第一个 [0]：这是在去除批次（Batch）维度，也就是取出了这批数据中的第一句话。此时形状变成了 [sequence_length, hidden_size]。

第二个 [0, :]：这是一个切片操作。它取出了这句话里的第 0 个词元（所有 768 个维度的特征）。在 BERT 的设定里，句子的第 0 个词元永远是特殊的分类标志 [CLS]。

2. 通过全连接层 (Dense Layer)
代码片段：bert.pooler.dense(...)

你把刚刚提取出来的 [CLS] 词元特征，输入到了 pooler 模块的 dense 层中。

这就是一个简单的线性层（Linear Layer），它会将这个 768 维的向量乘上 pooler 专属的权重矩阵并加上偏置，进行一次线性映射。

In [ ]:
my_output = bert.pooler.activation(bert.pooler.dense(output['last_hidden_state'][0][0,:]))

In [ ]:
my_output.shape

torch.Size([768])

In [ ]:
torch.equal(my_output,output['pooler_output'][0])

True

# 所谓 pooler_output，在底层仅仅就是提取了 last_hidden_state 的第一个词元 [CLS]，并让它过了一个全连接层和 Tanh 激活函数而已。